In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Set random seed for reproducibility
np.random.seed(42)

# Set up date range for 1 year (3 shifts per day)
start_date = datetime(2025, 1, 1)
end_date = datetime(2025, 12, 31)
delta = end_date - start_date

shifts = ['Morning', 'Evening', 'Night']
records = []

current_date = start_date
while current_date <= end_date:
    for shift in shifts:
        # Generate baseline values with slight random variation (normal distribution)
        biuret = np.random.normal(loc=0.75, scale=0.08)       # Limit: < 1.0%
        moisture = np.random.normal(loc=0.22, scale=0.03)     # Limit: < 0.3%
        prill_size = np.random.normal(loc=93.5, scale=2.1)    # Limit: > 90%
        ammonia_purity = np.random.normal(loc=99.7, scale=0.1) # Limit: > 99.5%
        water_ph = np.random.normal(loc=9.0, scale=0.25)       # Limit: 8.5 - 9.5
        
        # Inject periodic "Process Upsets" (OOS Events) to make data realistic
        # Scenario A: High ambient humidity during July/August (Monsoon season in Pakistan)
        if current_date.month in [7, 8] and np.random.rand() > 0.7:
            moisture += 0.12  # Pushes moisture over the limit
            
        # Scenario B: Reactor catalyst aging (Gradual drift in Biuret and Ammonia Purity)
        if current_date.month == 11 and np.random.rand() > 0.8:
            biuret += 0.28
            ammonia_purity -= 0.35
            
        # Scenario C: Random boiler chemical dosing failure
        if np.random.rand() > 0.98:
            water_ph -= 1.2 # Severe drop into acidic range (corrosion risk)

        records.append({
            'Timestamp': current_date.strftime('%Y-%m-%d'),
            'Shift': shift,
            'Analyst_ID': np.random.choice(['AN-01', 'AN-02', 'AN-03', 'AN-04']),
            'Urea_Biuret_Pct': round(max(0.1, biuret), 3),
            'Urea_Moisture_Pct': round(max(0.01, moisture), 3),
            'Urea_Prill_Size_Pct': round(min(100, max(50, prill_size)), 2),
            'Ammonia_Purity_Pct': round(min(100, max(90, ammonia_purity)), 3),
            'Boiler_Water_pH': round(max(1, min(14, water_ph)), 2)
        })
        
    current_date += timedelta(days=1)

# Convert to DataFrame and save
df = pd.DataFrame(records)
df.to_csv('fertilizer_lab_raw_data.csv', index=False)
print(f"Successfully generated {len(df)} rows of synthetic laboratory data!")

Successfully generated 1095 rows of synthetic laboratory data!
